# Phase 2: beam search + execution-guided reranking (Kaggle)
**Before running:** upload the NEW `finqa-diagnosis` zip as a Kaggle dataset (or add a new version of `project-atinedl`) so it contains `scripts/09_beam_decode.py`, `scripts/10_rerank_eval.py` and `src/finqadiag/rerank.py`. Accelerator = GPU, Internet = On. Use **Save Version -> Save & Run All (Commit)**.
Total time: about 1 h 10 min (55 min training, ~10 min beam decoding).

In [ ]:
import glob, os, shutil
hit = glob.glob("/kaggle/input/**/scripts/09_beam_decode.py", recursive=True)
assert hit, "scripts/09_beam_decode.py not found: attach the NEW dataset version"
src = os.path.dirname(os.path.dirname(hit[0]))
print("source:", src)
shutil.rmtree("/kaggle/working/finqa-diagnosis", ignore_errors=True)
shutil.copytree(src, "/kaggle/working/finqa-diagnosis")
%cd /kaggle/working/finqa-diagnosis
!pip -q install transformers tokenizers tqdm matplotlib pytest
import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
!bash scripts/00_download_data.sh
!python -m pytest -q

## Smoke test (2-3 min): train 64 examples, beam-decode, rerank. Must finish without errors.

In [ ]:
!python scripts/06_train_generator.py --limit 64 --epochs 1 --retrieval released --out runs/smoke
!python scripts/09_beam_decode.py --ckpt runs/smoke --retrieval released --beam 4 --limit 64
!python scripts/10_rerank_eval.py --beams runs/smoke --name smoke

## Real run: train the generator (seed 0, released retrieval), keep the checkpoint

In [ ]:
!python scripts/06_train_generator.py --model roberta-base --retrieval released --seed 0 --out runs/gen_beam

## Beam decode (beam 16) and rerank

In [ ]:
!python scripts/09_beam_decode.py --ckpt runs/gen_beam --retrieval released --beam 16
!python scripts/10_rerank_eval.py --beams runs/gen_beam --name released

## Package (download `phase2.zip` from the Output tab and attach it in the chat)

In [ ]:
!zip -qr phase2.zip runs/gen_beam results/rerank_*.json -x '*.pt' && ls -la phase2.zip